<a href="https://colab.research.google.com/github/ttktjmt/mjswan/blob/main/examples/colab/anymal_c_velocity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🦢 mjswan Tutorial with anymal_c_velocity**

This notebook is an example of visualization with mjswan for the trained policy with mjlab [anymal_c_velocity](https://github.com/mujocolab/anymal_c_velocity)

<img src="https://github.com/mujocolab/anymal_c_velocity/blob/main/teaser.gif?raw=true" width="300">

## **⚙️ Setup**

In [ ]:
!pip install uv -q
!uv pip install "git+https://github.com/mujocolab/anymal_c_velocity.git" "git+https://github.com/ttktjmt/mjswan.git" --system -q

## **✔️ Sanity Check (optional)**

In [ ]:
import re
import subprocess
import sys

process = subprocess.Popen(
    [
        "uv",
        "run",
        "play",
        "Mjlab-Velocity-Flat-Anymal-C",
        "--agent",
        "random",
        "--num_envs",
        "4",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1,
)

detected_port = None

for line in process.stdout:
    print(line, end="")
    sys.stdout.flush()

    # Extract port number from viser output
    port_match = re.search(r":(\d{4})", line)
    if port_match and "viser" in line.lower():
        detected_port = int(port_match.group(1))
        print("\n" + "=" * 34)
        print(f"✅ Server is running on port {detected_port}!")
        print("=" * 34)
        break

In [ ]:
from google.colab import output

port = detected_port if detected_port else 8081
output.serve_kernel_port_as_iframe(port=port, height=600)  # Wait few minutes

## **📚 Train the anymal_c_velocity Policy**

### **🔑 WandB Setup**

In [ ]:
# Online WandB (if you have an account)
!wandb login

# Local WandB
# !wandb offline

### **🏋️ Train the policy**

This will take a couple of hours

In [ ]:
!uv run train Mjlab-Velocity-Flat-Anymal-C --env.scene.num-envs 4096 --agent.max-iterations 2_001

## **♺ Functions to Get Assets**

In [ ]:
import json
import tempfile
from pathlib import Path

import mujoco
import onnx
import torch
from mjlab.scene import Scene
from mjlab.tasks.registry import load_env_cfg

# 1. Scene Spec


def get_mjlab_scene_spec(task_id: str) -> mujoco.MjSpec:
    env_cfg = load_env_cfg(task_id)
    env_cfg.scene.num_envs = 1
    scene = Scene(env_cfg.scene, device="cpu")

    return scene.spec


# 2. ONNX ModelProto


def download_latest_pt(run) -> Path:
    latest_file = None
    latest_step = -1

    for f in run.files():
        if f.name.endswith(".pt"):
            # "model_2000.pt" -> 2000
            try:
                step = int(f.name.replace("model_", "").replace(".pt", ""))
            except ValueError:
                continue
            if step > latest_step:
                latest_step = step
                latest_file = f

    if latest_file is None:
        raise RuntimeError("No .pt file found")

    print(f"Downloading {latest_file.name}...")

    tmp = tempfile.mkdtemp()
    return Path(latest_file.download(root=tmp, replace=True).name)


def get_onnx_policy(run) -> onnx.ModelProto:
    # Find and download the onnx file
    onnx_file = next((f for f in run.files() if f.name.endswith(".onnx")), None)
    if onnx_file is None:
        raise RuntimeError("No .onnx file found")

    tmp = tempfile.mkdtemp()
    onnx_path = Path(onnx_file.download(root=tmp, replace=True).name)
    onnx_model = onnx.load(str(onnx_path))
    print(f"Downloading {onnx_file.name} to {str(onnx_path)}")

    # Load the latest pt weight
    pt_path = download_latest_pt(run)
    actor_state_dict = torch.load(pt_path, map_location="cpu")["actor_state_dict"]

    # Override weight
    for initializer in onnx_model.graph.initializer:
        if initializer.name in actor_state_dict:
            new_init = onnx.numpy_helper.from_array(
                actor_state_dict[initializer.name].numpy(), name=initializer.name
            )
            onnx_model.graph.initializer.remove(initializer)
            onnx_model.graph.initializer.append(new_init)

    return onnx_model


# 3. Policy Config JSON


def get_policy_config(onnx_model: onnx.ModelProto) -> str:
    """Generate a policy config JSON from metadata embedded in the ONNX model.

    mjlab's VelocityOnPolicyRunner embeds joint_names, joint_stiffness,
    joint_damping, default_joint_pos, and action_scale into the ONNX file at
    export time. This function reads those values and pairs them with the
    standard obs_config for the mjlab velocity flat task (no height_scan).
    """

    def parse_floats(s: str) -> list[float]:
        return [float(v.strip()) for v in s.split(",")]

    meta = {prop.key: prop.value for prop in onnx_model.metadata_props}

    # mjlab's entity.joint_names strips the "robot/" scene prefix via
    # split("/")[-1], so the ONNX metadata contains bare names like "LF_HAA".
    # The MjSpec returned by get_mjlab_scene_spec() attaches the robot with
    # prefix="robot/", so the actual joint names in the scene are "robot/LF_HAA".
    # We must re-add the prefix here so mjswan can look up joints in the model.
    joint_names = [f"robot/{v.strip()}" for v in meta["joint_names"].split(",")]
    default_joint_pos = parse_floats(meta["default_joint_pos"])
    joint_stiffness = parse_floats(meta["joint_stiffness"])
    joint_damping = parse_floats(meta["joint_damping"])
    action_scale_raw = meta["action_scale"]
    try:
        action_scale: float | list[float] = float(action_scale_raw)
    except ValueError:
        action_scale = parse_floats(action_scale_raw)

    # Actual ONNX inputs (exclude weight initializers)
    initializer_names = {init.name for init in onnx_model.graph.initializer}
    in_keys = [
        inp.name for inp in onnx_model.graph.input if inp.name not in initializer_names
    ]

    # Normalize output keys to the standard mjswan convention.
    # runtime.ts looks for result.action or result.policy (not result.actions).
    # OnnxModule uses out_keys as aliases for ONNX outputs (mapped by position),
    # so "actions" → "action" ensures the runtime finds the tensor correctly.
    _OUT_KEY_MAP = {"actions": "action"}
    out_keys = [_OUT_KEY_MAP.get(out.name, out.name) for out in onnx_model.graph.output]

    # obs_config for mjlab velocity flat task.
    # Matches the actor_terms order defined in velocity_env_cfg.py
    # (height_scan is excluded for flat terrain).
    #
    # JointPos: pos_steps must be set explicitly to [0] (current step only).
    #   The frontend default is [0,1,2,3,4,8] (6 steps → 72 dims), but
    #   mjlab's joint_pos_rel uses a single timestep → 12 dims.
    obs_config = {
        in_keys[0]: [
            {"name": "BaseLinearVelocity"},
            {"name": "BaseAngularVelocity"},
            {"name": "ProjectedGravityB"},
            {
                "name": "JointPos",
                "pos_steps": [0],
                "subtract_default": True,
                "scale": 1.0,
            },
            {"name": "JointVelocities"},
            {"name": "PrevActions", "history_steps": 1},
            {"name": "SimpleVelocityCommand"},
        ]
    }

    config = {
        "policy_joint_names": joint_names,
        "default_joint_pos": default_joint_pos,
        "obs_config": obs_config,
        "action_scale": action_scale,
        "stiffness": joint_stiffness,
        "damping": joint_damping,
        "control_type": "joint_position",
        "onnx": {
            "meta": {
                "in_keys": in_keys,
                "out_keys": out_keys,
            }
        },
    }

    config_path = Path(tempfile.mkdtemp()) / "anymal_c_velocity.json"
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)

    return str(config_path)

## **🦢 Generate the mjswan App**

In [ ]:
import wandb

import mjswan

task_id = "Mjlab-Velocity-Flat-Anymal-C"

if wandb.run is not None:
    run = wandb.run
    print(run.name)
else:
    print("Not logged into WandB")
    api = wandb.Api()
    run = api.run("ttktjmt-org/mjlab/28eqz4yg")

APP_DIR = Path("/content/mjswan/app")


builder = mjswan.Builder()
project = builder.add_project(name="anymal_c_velocity")

anymal_c_scene = project.add_scene(spec=get_mjlab_scene_spec(task_id), name=task_id)

policy = get_onnx_policy(run)

anymal_c_scene.add_policy(
    name="anymal_c_velocity",
    policy=policy,
    config_path=get_policy_config(policy),
).add_velocity_command(
    lin_vel_x=(-2.0, 2.0),
    lin_vel_y=(-2.0, 2.0),
    default_lin_vel_x=0.2,
    default_lin_vel_y=0.2,
)

app = builder.build(output_dir=APP_DIR)

## **🌐 Serve Visualization**

In [ ]:
import http.server
import socketserver
import threading

PORT = 8000


class Handler(http.server.SimpleHTTPRequestHandler):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, directory=APP_DIR, **kwargs)


def start_server():
    with socketserver.TCPServer(("", PORT), Handler) as httpd:
        print(f"Serving at port {PORT}")
        httpd.serve_forever()


thread = threading.Thread(target=start_server, daemon=True)
thread.start()

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(PORT, height="700")